In [1]:
import MetaTrader5 as mt5
import time
from datetime import datetime
import pytz
import pandas as pd

#### Connection to MT5 account

In [2]:
if not mt5.initialize():
    print("initialize() failed, error code =", mt5.last_error())
    quit()

#### Disconnecting

In [7]:
mt5.shutdown()

True

#### Account level information

In [10]:
# !profit shows pending, not overall history
account_info = mt5.account_info()
if account_info != None:
    # display trading account data as is
    print(account_info)
    # display data about the trading account in the form of a dictionary
    print("Show account_info()._asdict():")
    account_info_dict = mt5.account_info()._asdict()
    for prop in account_info_dict:
        print("  {}={}".format(prop, account_info_dict[prop]))

AccountInfo(login=5030627923, trade_mode=0, leverage=100, limit_orders=200, margin_so_mode=0, trade_allowed=True, trade_expert=True, margin_mode=2, currency_digits=2, fifo_close=False, balance=99999.91, credit=0.0, profit=-0.15, equity=99999.76, margin=10.0, margin_free=99989.76, margin_level=999997.5999999999, margin_so_call=50.0, margin_so_so=30.0, margin_initial=0.0, margin_maintenance=0.0, assets=0.0, liabilities=0.0, commission_blocked=0.0, name='Maximilian Pazze', server='MetaQuotes-Demo', currency='EUR', company='MetaQuotes Ltd.')
Show account_info()._asdict():
  login=5030627923
  trade_mode=0
  leverage=100
  limit_orders=200
  margin_so_mode=0
  trade_allowed=True
  trade_expert=True
  margin_mode=2
  currency_digits=2
  fifo_close=False
  balance=99999.91
  credit=0.0
  profit=-0.15
  equity=99999.76
  margin=10.0
  margin_free=99989.76
  margin_level=999997.5999999999
  margin_so_call=50.0
  margin_so_so=30.0
  margin_initial=0.0
  margin_maintenance=0.0
  assets=0.0
  liab

#### Financial Instruments - newest ticks

In [5]:
# market watch
selected=mt5.symbol_select("GBPUSD", True)
if not selected:
    print("Failed to select GBPUSD")
    mt5.shutdown()
    quit()
# display the last tick of the GBPUSD symbol as a tuple
lasttick = mt5.symbol_info_tick("GBPUSD")
print(lasttick)
# display the values of the tick fields in the form of a dictionary
print("Show symbol_info_tick(\"GBPUSD\")._asdict():")
symbol_info_tick_dict = lasttick._asdict()
for prop in symbol_info_tick_dict:
    print("  {}={}".format(prop, symbol_info_tick_dict[prop]))

Tick(time=1728483425, bid=1.30944, ask=1.3094999999999999, last=0.0, volume=0, time_msc=1728483425974, flags=1030, volume_real=0.0)
Show symbol_info_tick("GBPUSD")._asdict():
  time=1728483425
  bid=1.30944
  ask=1.3094999999999999
  last=0.0
  volume=0
  time_msc=1728483425974
  flags=1030
  volume_real=0.0


#### Financial Instruments - order book

In [9]:
if mt5.market_book_add('EURUSD'):
    # run 10 times a loop to read data from the order book
    for i in range(10):
        # get the contents of the order book
        items = mt5.market_book_get('EURUSD')
        # display the entire order book in one line as is
        print(items)
        # now display each price level separately in the form of a dictionary, for clarity
        for it in items or []:
            print(it._asdict())
        # let's pause for 5 seconds before the next request for data from the order book
        time.sleep(5)
        # unsubscribe to order book changes
    mt5.market_book_release('EURUSD')
else:
    print("mt5.market_book_add('EURUSD') failed, error code =", mt5.last_error())

()
()
()
()
(BookInfo(type=1, price=1.09667, volume=10, volume_dbl=10.0), BookInfo(type=1, price=1.09664, volume=20, volume_dbl=20.0), BookInfo(type=1, price=1.09662, volume=10, volume_dbl=10.0), BookInfo(type=1, price=1.0966, volume=7, volume_dbl=7.5), BookInfo(type=1, price=1.0965799999999999, volume=2, volume_dbl=2.5), BookInfo(type=2, price=1.09654, volume=2, volume_dbl=2.5), BookInfo(type=2, price=1.09652, volume=2, volume_dbl=2.5), BookInfo(type=2, price=1.09651, volume=5, volume_dbl=5.0), BookInfo(type=2, price=1.09649, volume=10, volume_dbl=10.0), BookInfo(type=2, price=1.09647, volume=30, volume_dbl=30.0))
{'type': 1, 'price': 1.09667, 'volume': 10, 'volume_dbl': 10.0}
{'type': 1, 'price': 1.09664, 'volume': 20, 'volume_dbl': 20.0}
{'type': 1, 'price': 1.09662, 'volume': 10, 'volume_dbl': 10.0}
{'type': 1, 'price': 1.0966, 'volume': 7, 'volume_dbl': 7.5}
{'type': 1, 'price': 1.0965799999999999, 'volume': 2, 'volume_dbl': 2.5}
{'type': 2, 'price': 1.09654, 'volume': 2, 'volume_

#### Financial Instruments - candle information

In [16]:
# set the timezone to UTC (-2 hrs. to CET)
timezone = pytz.timezone("Etc/UTC")
# create a datetime object in the UTC timezone so that the local timezone offset is not applied
utc_from = datetime(2024, 10, 9, tzinfo = timezone)
# get 3 bars from EURUSD H1 starting from 09/10/2024 in the UTC timezone
rates = mt5.copy_rates_from("EURUSD", mt5.TIMEFRAME_H1, utc_from, 3)
# display each element of the received data (tuple)
for rate in rates:
    #dt object, Open, High, Low, Close, Volume, Spread, own volume
    print(rate)

(1728424800, 1.09721, 1.09781, 1.09694, 1.09776, 1078, 4, 0)
(1728428400, 1.09775, 1.09802, 1.09713, 1.09721, 542, 3, 0)
(1728432000, 1.0977, 1.09799, 1.09686, 1.09786, 580, 4, 0)


#### Financial Instruments - historical candle information

In [3]:
# set the timezone to UTC
timezone = pytz.timezone("Etc/UTC")
# create datetime objects in UTC timezone so that local timezone offset is not applied
utc_from = datetime(2024, 1, 1, tzinfo=timezone)
utc_to = datetime(2024, 10, 8, tzinfo=timezone)

rates = mt5.copy_rates_range("EURUSD", mt5.TIMEFRAME_D1, utc_from, utc_to)

# create a DataFrame from the received data
rates_frame = pd.DataFrame(rates)
# convert time from number of seconds to datetime format
rates_frame['time'] = pd.to_datetime(rates_frame['time'], unit = 's')

# output data
rates_frame

,time,open,high,low,close,tick_volume,spread,real_volume
0,2024-01-02,1.10437,1.10451,1.09374,1.09407,205171,0,0
1,2024-01-03,1.09425,1.09655,1.08925,1.09212,234801,0,0
2,2024-01-04,1.09205,1.09723,1.09056,1.09435,195260,0,0
3,2024-01-05,1.09443,1.09988,1.08769,1.09408,263575,0,0
4,2024-01-08,1.09331,1.09789,1.09225,1.09496,183411,0,0
...,...,...,...,...,...,...,...,...
196,2024-10-02,1.10628,1.10827,1.10326,1.10432,38465,2,0
197,2024-10-03,1.10429,1.10491,1.10080,1.10306,40294,1,0
198,2024-10-04,1.10334,1.10396,1.09512,1.09737,38964,0,0
199,2024-10-07,1.09633,1.09867,1.09541,1.09739,39848,1,0


#### Opening orders

In [5]:
symbol = "EURUSD"
symbol_info = mt5.symbol_info(symbol)
if symbol_info is None:
    print(symbol, "not found, can not trade")
    mt5.shutdown()
    quit()

# if the symbol is not available in the Market Watch, add it
if not symbol_info.visible:
    print(symbol, "is not visible, trying to switch on")
    if not mt5.symbol_select(symbol, True):
        print("symbol_select({}) failed, exit", symbol)
        mt5.shutdown()
        quit()

# let's prepare the request structure for the purchase
lot = 0.01
point = mt5.symbol_info(symbol).point
price = mt5.symbol_info_tick(symbol).ask
deviation = 20
request = \
    {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": symbol,
        "volume": lot,
        "type": mt5.ORDER_TYPE_BUY,
        "price": price,
        "sl": price - 100 * point,
        "tp": price + 100 * point,
        "deviation": deviation,
        "magic": 234000,
        "comment": "python script open",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_RETURN,
    }

# send a trade request to open a position
result = mt5.order_send(request)
# check the execution result
print("1. order_send(): by {} {} lots at {}".format(symbol, lot, price));
if result.retcode != mt5.TRADE_RETCODE_DONE:
    print("2. order_send failed, retcode={}".format(result.retcode))
    # request the result as a dictionary and display it element by element
    result_dict = result._asdict()
    for field in result_dict.keys():
        print("   {}={}".format(field, result_dict[field]))
        # if this is the structure of a trade request, then output it element by element as well
        if field == "request":
            traderequest_dict = result_dict[field]._asdict()
            for tradereq_filed in traderequest_dict:
                print("traderequest: {}={}".format(tradereq_filed, traderequest_dict[tradereq_filed]))

print("2. order_send done, ", result)
print("   opened position with POSITION_TICKET={}".format(result.order))

1. order_send(): by EURUSD 0.01 lots at 1.0955300000000001
2. order_send done,  OrderSendResult(retcode=10009, deal=51654546266, order=51686594508, volume=0.01, price=1.0955300000000001, bid=1.09548, ask=1.0955300000000001, comment='Request executed', request_id=1821896447, retcode_external=0, request=TradeRequest(action=1, magic=234000, order=0, symbol='EURUSD', volume=0.01, price=1.0955300000000001, stoplimit=0.0, sl=1.0945300000000002, tp=1.09653, deviation=20, type=0, type_filling=2, type_time=0, expiration=0, comment='python script open', position=0, position_by=0))
   opened position with POSITION_TICKET=51686594508


#### Closing orders

In [6]:
 # create a request to close 
position_id = result.order
price = mt5.symbol_info_tick(symbol).bid
request = \
    {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": symbol,
        "volume": lot,
        "type": mt5.ORDER_TYPE_SELL,
        "position": position_id,
        "price": price,
        "deviation": deviation,
        "magic": 234000,
        "comment": "python script close",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_RETURN,
    }
# send a trade request to close the position
result = mt5.order_send(request)
# check the execution result
print("3. close position #{}: sell {} {} lots at {}".format(position_id, symbol, lot, price));
if result.retcode != mt5.TRADE_RETCODE_DONE:
    print("4. order_send failed, retcode={}".format(result.retcode))
    print("   result", result)
else:
    print("4. position #{} closed, {}".format(position_id, result))
    # request the result as a dictionary and display it element by element
    result_dict = result._asdict()
    for field in result_dict.keys():
        print("   {}={}".format(field, result_dict[field]))
        # if this is the structure of a trade request, then output it element by element as well
        if field == "request":
            traderequest_dict = result_dict[field]._asdict()
            for tradereq_filed in traderequest_dict:
                print("traderequest: {}={}".format(tradereq_filed, traderequest_dict[tradereq_filed]))

3. close position #51686594508: sell EURUSD 0.01 lots at 1.09543
4. position #51686594508 closed, OrderSendResult(retcode=10009, deal=51654585802, order=51686634051, volume=0.01, price=1.09543, bid=1.09543, ask=1.09547, comment='Request executed', request_id=1821896448, retcode_external=0, request=TradeRequest(action=1, magic=234000, order=0, symbol='EURUSD', volume=0.01, price=1.09543, stoplimit=0.0, sl=0.0, tp=0.0, deviation=20, type=1, type_filling=2, type_time=0, expiration=0, comment='python script close', position=51686594508, position_by=0))
   retcode=10009
   deal=51654585802
   order=51686634051
   volume=0.01
   price=1.09543
   bid=1.09543
   ask=1.09547
   comment=Request executed
   request_id=1821896448
   retcode_external=0
   request=TradeRequest(action=1, magic=234000, order=0, symbol='EURUSD', volume=0.01, price=1.09543, stoplimit=0.0, sl=0.0, tp=0.0, deviation=20, type=1, type_filling=2, type_time=0, expiration=0, comment='python script close', position=51686594508,

#### Looking up open orders

In [3]:
positions = mt5.positions_get(symbol="EURUSD")

if positions is None:
    print("No positions on EURUSD")
elif len(positions) > 0:
    print("Total positions on EURUSD =", len(positions))
    for position in positions:
        print(position)
else:
    print("No open positions on EURUSD")  # In case there are zero open positions


No open positions on EURUSD


In [4]:
print(len(positions))

0


#### Order History

In [8]:
# get the number of orders in the history for the period (total and *EUR*)
from_date = datetime(2024, 10, 1)
to_date = datetime.now()
total = mt5.history_orders_total(from_date, to_date)
history_orders=mt5.history_orders_get(from_date, to_date, group="*EUR*")
# print(history_orders)
if history_orders == None:
    print("No history orders with group=\"*EUR*\", error code={}".format(mt5.last_error()))
else :
    print("history_orders_get({}, {}, group=\"*EUR*\")={} of total {}".format(from_date, to_date, len(history_orders), total))

history_orders_get(2024-10-01 00:00:00, 2024-10-09 19:43:18.420901, group="*EUR*")=2 of total 2


In [12]:
 # set the time range
from_date = datetime(2024, 10, 1)
to_date = datetime.now()

# get trades for symbols 
deals = mt5.history_deals_get(from_date, to_date)
if deals == None:
    print("No deals, error code={}".format(mt5.last_error()))
elif len(deals) > 0:
    print("history_deals_get(from_date, to_date, group=\"*,!*EUR*,!*GBP*\") =",
          len(deals))
    # display all received deals as they are
    for deal in deals:
        print("  ",deal)
        # display these trades as a table using pandas.DataFrame
    df = pd.DataFrame(list(deals), columns = deals[0]._asdict().keys())
    df['time'] = pd.to_datetime(df['time'], unit='s')
    df.drop(['time_msc','commission','fee'], axis = 1, inplace = True)

history_deals_get(from_date, to_date, group="*,!*EUR*,!*GBP*") = 3
   TradeDeal(ticket=51650319826, order=0, time=1728403861, time_msc=1728403861343, type=2, entry=0, magic=0, position_id=0, reason=0, volume=0.0, price=0.0, commission=0.0, swap=0.0, profit=100000.0, fee=0.0, symbol='', comment='', external_id='')
   TradeDeal(ticket=51654546266, order=51686594508, time=1728489437, time_msc=1728489437475, type=0, entry=0, magic=234000, position_id=51686594508, reason=3, volume=0.01, price=1.0955300000000001, commission=0.0, swap=0.0, profit=0.0, fee=0.0, symbol='EURUSD', comment='python script op', external_id='')
   TradeDeal(ticket=51654585802, order=51686634051, time=1728489899, time_msc=1728489899080, type=1, entry=1, magic=234000, position_id=51686594508, reason=3, volume=0.01, price=1.09543, commission=0.0, swap=0.0, profit=-0.09, fee=0.0, symbol='EURUSD', comment='python script cl', external_id='')


In [13]:
df

,ticket,order,time,type,entry,magic,position_id,reason,volume,price,swap,profit,symbol,comment,external_id
0,51650319826,0,2024-10-08 16:11:01,2,0,0,0,0,0.00,0.00000,0.0,100000.00,,,
1,51654546266,51686594508,2024-10-09 15:57:17,0,0,234000,51686594508,3,0.01,1.09553,0.0,0.00,EURUSD,python script op,
2,51654585802,51686634051,2024-10-09 16:04:59,1,1,234000,51686594508,3,0.01,1.09543,0.0,-0.09,EURUSD,python script cl,
